# CelebA-HQ — Stage 5 metadata-only loader/split binding

Run with **Accelerator = None**. Attach the DIRE input Dataset, the code Dataset version containing `dire_modal/`, and the Stage 3 v2 split artifact (or keep its JSON under `/kaggle/working`). This cell validates split IDs, paths, identities, and counts; it does not compare per-run metadata fingerprints, open image contents, create tensors, use GPU, or train. The complete metadata scan can take time; the latest loader package prints progress during path indexing. The attached code package must include the fingerprint-disabled loader; older package versions are detected and rejected with an update instruction.

The output report is written only to `/kaggle/working/celebahq-data-binding-report.json`; the mounted input remains read-only. Do not index/iterate the returned datasets in this stage.

In [ ]:
from pathlib import Path
import inspect
import sys

INPUT_ROOT = Path('/kaggle/input')
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
SPLIT_ARTIFACT = Path('/kaggle/working/celebahq-validation-split-v2-81dcda99aeb9.json')

# Kaggle mounts attached Datasets but does not add their Python packages to sys.path.
# Search only mount roots (and optional src/), never recurse through the image tree.
mount_roots = []
for mount in INPUT_ROOT.iterdir():
    if mount.name == 'datasets' and mount.is_dir():
        for owner in mount.iterdir():
            if owner.is_dir():
                mount_roots.extend(dataset for dataset in owner.iterdir() if dataset.is_dir())
    elif mount.is_dir():
        mount_roots.append(mount)
code_roots = [
    root for mount in mount_roots for root in (mount, mount / 'src')
    if (root / 'dire_modal' / '__init__.py').is_file()
    and (root / 'dire_modal' / 'kaggle_celebahq_data.py').is_file()
]
if len(code_roots) != 1:
    listing = {str(mount): [child.name for child in mount.iterdir()][:12] for mount in mount_roots}
    raise FileNotFoundError(f'Expected one attached code package containing dire_modal; found {code_roots}. Mount listing: {listing}')
CODE_DATASET_ROOT = code_roots[0]
if not DIRE_ROOT.is_dir():
    raise FileNotFoundError(f'Missing DIRE input root: {DIRE_ROOT}')
if not SPLIT_ARTIFACT.is_file():
    raise FileNotFoundError(f'Missing Stage 3 v2 split artifact: {SPLIT_ARTIFACT}')

sys.path.insert(0, str(CODE_DATASET_ROOT))
from dire_modal.kaggle_celebahq_data import build_celebahq_datasets

BUILDER_SOURCE_PATH = Path(inspect.getsourcefile(build_celebahq_datasets)).resolve()
if CODE_DATASET_ROOT.resolve() not in BUILDER_SOURCE_PATH.parents:
    raise RuntimeError(
        f'Imported loader came from {BUILDER_SOURCE_PATH}, outside the selected code Dataset '
        f'{CODE_DATASET_ROOT}. Restart the kernel and attach only the intended code Dataset.'
    )
builder_source = inspect.getsource(build_celebahq_datasets)
if '_source_fingerprint(' in builder_source or 'metadata fingerprint differs' in builder_source:
    raise RuntimeError(
        'The attached Kaggle code Dataset is outdated: its loader still compares the per-mount '
        'metadata fingerprint. Publish/update the code-only dire-modal Dataset with the current '
        'src/dire_modal/kaggle_celebahq_data.py, attach that version, then restart the kernel. '
        f'Loaded file: {BUILDER_SOURCE_PATH}'
    )

print('Code package root added to sys.path:', CODE_DATASET_ROOT)
print('Active binding loader:', BUILDER_SOURCE_PATH)
print('DIRE root:', DIRE_ROOT)
print('Split artifact:', SPLIT_ARTIFACT)

In [ ]:
from collections import Counter
import inspect
import json

def progress(message: str) -> None:
    print('[binding]', message, flush=True)

builder_kwargs = {}
if 'progress_callback' in inspect.signature(build_celebahq_datasets).parameters:
    builder_kwargs['progress_callback'] = progress
else:
    print('Attached code package lacks progress callback; scan still runs without intermediate progress.')

bundle = build_celebahq_datasets(DIRE_ROOT, SPLIT_ARTIFACT, **builder_kwargs)

train_counts = Counter(record.label for record in bundle.train.records)
validation_counts = Counter(record.label for record in bundle.validation.records)
test_counts = {name: len(dataset) for name, dataset in bundle.test_by_generator.items()}

expected_train = {0: 25_200, 1: 36_000}
expected_validation = {0: 2_800, 1: 4_000}
# Inventory counts previously reported for this mounted CelebA-HQ test tree.
expected_test = {
    'real': 1_000, 'dalle2': 500, 'if': 1_000,
    'midjourney': 100, 'stargan': 1_000, 'sdv2': 1_000,
}

assert dict(train_counts) == expected_train, (dict(train_counts), expected_train)
assert dict(validation_counts) == expected_validation, (dict(validation_counts), expected_validation)
assert test_counts == expected_test, (test_counts, expected_test)
assert bundle.validation_id_list_sha256 == (
    'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
)

report = {
    'status': 'pass',
    'protocol': 'celebahq-content-disjoint-exact-count-hash-rank-v2',
    'metadata_fingerprint_check': 'not_performed',
    'validation_id_list_sha256': bundle.validation_id_list_sha256,
    'train_count_by_label': {str(k): v for k, v in sorted(train_counts.items())},
    'validation_count_by_label': {str(k): v for k, v in sorted(validation_counts.items())},
    'test_count_by_generator': test_counts,
    'image_contents_read': False,
    'input_write_attempted': False,
    'gpu_workload_started': False,
    'training_started': False,
}
report_path = Path('/kaggle/working/celebahq-data-binding-report.json')
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved report:', report_path)

## Stage 6 — ImageNet checkpoint identity (read-only, CPU only)

Run only after the Stage 5 binding report says `pass`. This cell locates the cached `resnet50-19c8e357.pth` (or accepts `RESNET50_WEIGHTS_PATH`), reads its bytes once to record file size and full SHA-256, and checks the filename's 8-hex digest prefix. It does not deserialize the checkpoint, create a model, use a GPU, read dataset images, train, or write to mounted inputs. The report is saved under `/kaggle/working`. This is not training approval; seed/budget/retention still need review.


In [ ]:
from pathlib import Path
import hashlib
import json
import os

BINDING_REPORT_PATH = Path('/kaggle/working/celebahq-data-binding-report.json')
CHECKPOINT_REPORT_PATH = Path('/kaggle/working/celebahq-pretrained-checkpoint-report.json')
CHECKPOINT_BASENAME = 'resnet50-19c8e357.pth'
EXPECTED_SHA256_PREFIX = '19c8e357'

if not BINDING_REPORT_PATH.is_file():
    raise FileNotFoundError(f'Run Stage 5 binding first; report missing: {BINDING_REPORT_PATH}')
binding_report = json.loads(BINDING_REPORT_PATH.read_text(encoding='utf-8'))
if binding_report.get('status') != 'pass':
    raise RuntimeError(f'Stage 5 binding did not pass: {binding_report.get("status")!r}')
if binding_report.get('validation_id_list_sha256') != 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd':
    raise RuntimeError('Stage 5 report does not contain the locked validation ID-list hash.')

# Prefer an explicit variable/env override. Otherwise inspect only conventional
# PyTorch cache locations; do not recurse through Kaggle input datasets.
explicit_checkpoint = globals().get('RESNET50_WEIGHTS_PATH') or os.environ.get('RESNET50_WEIGHTS_PATH')
if explicit_checkpoint:
    candidates = [Path(explicit_checkpoint).expanduser()]
else:
    xdg_cache = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')).expanduser()
    torch_home = Path(os.environ.get('TORCH_HOME', xdg_cache / 'torch')).expanduser()
    candidates = [
        torch_home / 'hub' / 'checkpoints' / CHECKPOINT_BASENAME,
        Path.home() / '.cache' / 'torch' / 'hub' / 'checkpoints' / CHECKPOINT_BASENAME,
        Path('/kaggle/working/checkpoints') / CHECKPOINT_BASENAME,
    ]

found = sorted({path.resolve() for path in candidates if path.is_file()})
if not found:
    raise FileNotFoundError(
        f'Could not find {CHECKPOINT_BASENAME} in the standard cache locations. '
        'Set RESNET50_WEIGHTS_PATH to the exact existing checkpoint file and rerun this cell.'
    )
if len(found) != 1:
    raise RuntimeError(
        'Found multiple candidate ImageNet checkpoints; set RESNET50_WEIGHTS_PATH explicitly: '
        + ', '.join(map(str, found))
    )
checkpoint_path = found[0]
if checkpoint_path.name != CHECKPOINT_BASENAME:
    raise ValueError(f'Expected checkpoint basename {CHECKPOINT_BASENAME!r}, got {checkpoint_path.name!r}')

before = checkpoint_path.stat()
digest = hashlib.sha256()
with checkpoint_path.open('rb') as stream:
    for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
        digest.update(block)
after = checkpoint_path.stat()
stable_during_read = (before.st_size, before.st_mtime_ns, before.st_ino) == (
    after.st_size, after.st_mtime_ns, after.st_ino
)
sha256 = digest.hexdigest()
prefix_matches = sha256.startswith(EXPECTED_SHA256_PREFIX)
status = 'pass' if stable_during_read and prefix_matches else 'fail'
report = {
    'status': status,
    'checkpoint_path': str(checkpoint_path),
    'file_size_bytes': after.st_size,
    'sha256': sha256,
    'filename_sha256_prefix': EXPECTED_SHA256_PREFIX,
    'sha256_prefix_matches_filename': prefix_matches,
    'stable_during_read': stable_during_read,
    'binding_report': str(BINDING_REPORT_PATH),
    'checkpoint_deserialized': False,
    'dataset_image_contents_read': False,
    'gpu_workload_started': False,
    'training_started': False,
    'mounted_input_write_attempted': False,
    'note': 'Full SHA-256 recorded; filename prefix is only an 8-hex reference, not a full independent reference-hash verification.',
}
CHECKPOINT_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved checkpoint report:', CHECKPOINT_REPORT_PATH)
if status != 'pass':
    raise RuntimeError(f'Checkpoint verification failed; inspect {CHECKPOINT_REPORT_PATH}')


## Stage 7 — Training plan + fresh disk preflight (no training)

Run this after the Stage 5 binding cell and Stage 6 checkpoint audit pass, using the updated code Dataset that contains `kaggle_celebahq_training.py`. It validates the in-memory bundle, records the approved training plan, and checks current `/kaggle/working` free space against the total 3-seed reserve. It does not import PyTorch, inspect/use a GPU, read image bytes, deserialize weights, or call the trainer. The report is written only under `/kaggle/working`. This is not permission to train.


In [ ]:
from pathlib import Path
import inspect
import json
import shutil

PLAN_REPORT_PATH = Path('/kaggle/working/celebahq-training-plan.json')
BINDING_REPORT_PATH = Path('/kaggle/working/celebahq-data-binding-report.json')
CHECKPOINT_REPORT_PATH = Path('/kaggle/working/celebahq-pretrained-checkpoint-report.json')
EXPECTED_VALIDATION_SHA256 = 'c635c38a5888ef4e0196423dd8f342f412dc15ea633c17d835330c740d7dc9fd'
EXPECTED_CHECKPOINT_SHA256 = '19c8e3572231adff6824a2da93fd67b5986919a2e65f8b6007eab4edee220097'

if 'bundle' not in globals() or 'CODE_DATASET_ROOT' not in globals():
    raise RuntimeError('Run Stage 5 setup and binding cells first in this kernel.')
for required_report in (BINDING_REPORT_PATH, CHECKPOINT_REPORT_PATH):
    if not required_report.is_file():
        raise FileNotFoundError(f'Run the required prior stage first; report missing: {required_report}')
binding_report = json.loads(BINDING_REPORT_PATH.read_text(encoding='utf-8'))
checkpoint_report = json.loads(CHECKPOINT_REPORT_PATH.read_text(encoding='utf-8'))
if binding_report.get('status') != 'pass':
    raise RuntimeError('Stage 5 binding report is not pass.')
if binding_report.get('validation_id_list_sha256') != EXPECTED_VALIDATION_SHA256:
    raise RuntimeError('Stage 5 report has the wrong locked validation ID-list hash.')
if checkpoint_report.get('status') != 'pass':
    raise RuntimeError('Stage 6 checkpoint report is not pass.')
if checkpoint_report.get('sha256') != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError('Stage 6 checkpoint digest does not match the recorded full SHA-256.')
if checkpoint_report.get('checkpoint_deserialized') is not False:
    raise RuntimeError('Stage 6 report indicates the checkpoint was deserialized.')
checkpoint_path = Path(checkpoint_report.get('checkpoint_path', '')).expanduser()
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'ImageNet weights file is no longer present: {checkpoint_path}')
if checkpoint_path.stat().st_size != checkpoint_report.get('file_size_bytes'):
    raise RuntimeError('ImageNet checkpoint size changed since Stage 6 audit.')

try:
    from dire_modal.kaggle_celebahq_training import (
        CelebAHQTrainingConfig,
        build_training_plan,
    )
except ImportError as exc:
    raise RuntimeError(
        'The attached code Dataset lacks kaggle_celebahq_training.py. Publish/attach the updated '
        'code-only dire-modal Dataset, restart the kernel, and rerun Stages 5 and 6.'
    ) from exc
training_module_path = Path(inspect.getsourcefile(build_training_plan)).resolve()
if CODE_DATASET_ROOT.resolve() not in training_module_path.parents:
    raise RuntimeError(
        f'Training module came from outside the selected code Dataset: {training_module_path}'
    )

config = CelebAHQTrainingConfig()
plan = build_training_plan(
    bundle,
    config=config,
    weights_report_path=CHECKPOINT_REPORT_PATH,
)
working = Path('/kaggle/working')
free_bytes = shutil.disk_usage(working).free
suite_reserve_bytes = config.output_reserve_bytes * len(config.seeds)
status = 'ready_for_review' if free_bytes >= suite_reserve_bytes else 'needs_attention'
report = {
    **plan,
    'status': status,
    'training_module': str(training_module_path),
    'checkpoint': {
        'path': str(checkpoint_path.resolve()),
        'file_size_bytes': checkpoint_report['file_size_bytes'],
        'sha256': checkpoint_report['sha256'],
    },
    'disk_preflight': {
        'working_path': str(working.resolve()),
        'free_bytes': free_bytes,
        'reserve_per_seed_bytes': config.output_reserve_bytes,
        'seeds': len(config.seeds),
        'suite_reserve_bytes': suite_reserve_bytes,
        'passes': free_bytes >= suite_reserve_bytes,
    },
    'image_contents_read': False,
    'checkpoint_deserialized': False,
    'gpu_metadata_queried': False,
    'gpu_workload_started': False,
    'training_started': False,
    'mounted_input_write_attempted': False,
    'next_gate': 'review plan; do not call the training entry point without a separate explicit go-ahead',
}
PLAN_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved plan and disk report:', PLAN_REPORT_PATH)


## Stage 8 — Two-T4 metadata-only preflight

Run after Stage 7, with Kaggle Accelerator set to **GPU T4 x2**. This freshly queries device names/UUIDs/memory using `nvidia-smi`, requires exactly two distinct Tesla T4 devices, and writes a report under `/kaggle/working`. It does not import PyTorch, allocate tensors, start a GPU workload, read images/checkpoints, or train. A pass only confirms the two-device metadata gate; it is not authorization to start training.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess

PLAN_REPORT_PATH = Path('/kaggle/working/celebahq-training-plan.json')
GPU_REPORT_PATH = Path('/kaggle/working/celebahq-gpu-metadata-report.json')
if not PLAN_REPORT_PATH.is_file():
    raise FileNotFoundError(f'Run Stage 7 first: {PLAN_REPORT_PATH}')
plan = json.loads(PLAN_REPORT_PATH.read_text(encoding='utf-8'))
if plan.get('status') != 'ready_for_review' or plan.get('training_started') is not False:
    raise RuntimeError('Stage 7 plan is not in the expected no-training review state.')
command = [
    'nvidia-smi',
    '--query-gpu=name,uuid,memory.total,memory.free,driver_version',
    '--format=csv,noheader,nounits',
]
try:
    result = subprocess.run(command, capture_output=True, text=True, timeout=15, check=False)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or f'nvidia-smi exited {result.returncode}')
    devices = []
    for row in csv.reader(result.stdout.splitlines(), skipinitialspace=True):
        values = [value.strip() for value in row]
        if not values or not any(values):
            continue
        if len(values) != 5:
            raise RuntimeError(f'unexpected nvidia-smi row: {values}')
        devices.append({
            'name': values[0], 'uuid': values[1],
            'memory_total_mib': values[2], 'memory_free_mib': values[3],
            'driver_version': values[4],
        })
    valid = (
        len(devices) == 2
        and all('T4' in device['name'] and device['uuid'] for device in devices)
        and len({device['uuid'] for device in devices}) == 2
    )
    report = {
        'status': 'pass' if valid else 'needs_attention',
        'devices': devices,
        'expected': 'exactly two distinct NVIDIA Tesla T4 devices',
        'gpu_metadata_queried': True,
        'gpu_workload_started': False,
        'training_started': False,
        'image_contents_read': False,
        'checkpoint_deserialized': False,
        'mounted_input_write_attempted': False,
        'note': 'Metadata-only gate; does not authorize training.',
    }
except Exception as exc:
    report = {
        'status': 'needs_attention',
        'error': f'{type(exc).__name__}: {exc}',
        'gpu_metadata_queried': True,
        'gpu_workload_started': False,
        'training_started': False,
        'image_contents_read': False,
        'checkpoint_deserialized': False,
        'mounted_input_write_attempted': False,
        'note': 'No training was started; verify Kaggle Accelerator/runtime.',
    }
GPU_REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2))
print('Saved metadata-only GPU report:', GPU_REPORT_PATH)
